A European call option gives you: the right to buy an asset at a fixed price K (called the strike) at a fixed future date T (called expiry).

Payoff is max(S_T​−K,0) as you would not exercise for a loss. S_T is the price of the asset at the time of the expiry.

In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [67]:
config = {
    "S0": 1,
    "k": 1,
    "T": 1,
    "r": 0.05,
    "sigma": 0.2,
    "N": 10000
}

In [65]:
def monte_carlo_call(S0=1, K=1, T=1, r=0.05, sigma=0.2, N=10000, **kwargs):
    
    # Simulate S_T directly using closed form
    Z = np.random.normal(size=N)
    S_T = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    
    # Compute payoffs
    payoffs = np.maximum(S_T - K, 0)
    
    # Discount and average
    price = np.exp(-r * T) * np.mean(payoffs)
    
    return price

def asian_monte_carlo_call(S0=1, n=1000, K=1, T=1, r=0.05, sigma=0.2, N=10000, **kwargs):
    
    times = np.linspace(0, T, n)
    
    # Generate Wiener process paths
    dW = np.sqrt(times[1] - times[0]) * np.random.normal(size=(n-1, N))
    W = np.concatenate((np.zeros((1, N)), np.cumsum(dW, axis=0)), axis=0)
    
    # Evaluate closed form at each timestep for each path
    S = S0 * np.exp((r - 0.5 * sigma**2) * times[:, None] + sigma * W)
    
    # Average price for each path
    S_bar = np.mean(S, axis=0)
    
    # Compute payoffs
    payoffs = np.maximum(S_bar - K, 0)
    
    # Discount and average
    price = np.exp(-r * T) * np.mean(payoffs)
    
    return price

european = monte_carlo_call(**config)
asian = asian_monte_carlo_call(**config)

print(f"European call option: {european}")
print(f"Asian call option: {asian}")
print(f"Difference: {european - asian}")
print(f"Ratio: {asian / european}")

European call option: 0.10428985698240333
Asian call option: 0.05793917812574214
Difference: 0.046350678856661186
Ratio: 0.5555590908089761


## Variance Reduction

At each interval our value of W_t may overshoot or undershoot the true value. 

If W_t overshoots then -W_t undershoots by roughly the same amount. By averaging the payoffs together errors *cancel* out instead of accumulating.

In [68]:
def anti_monte_carlo_call(S0=1, K=1, T=1, r=0.05, sigma=0.2, N=10000, **kwargs):
    
    # Simulate S_T directly using closed form
    Z = np.random.normal(size=int(N/2))
    anti_Z = Z * -1

    S_T = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    anti_S_T = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * anti_Z)
    
    # Compute payoffs
    payoffs = np.concatenate([
        np.maximum(S_T - K, 0),
        np.maximum(anti_S_T - K, 0)
    ])
    
    # Discount and average
    price = np.exp(-r * T) * np.mean(payoffs)
    
    return price

def anti_asian_monte_carlo_call(S0=1, n=1000, K=1, T=1, r=0.05, sigma=0.2, N=10000, **kwargs):
    
    times = np.linspace(0, T, n)
    
    # Generate Wiener process paths
    dW = np.sqrt(times[1] - times[0]) * np.random.normal(size=(n-1, N))
    W = np.concatenate((np.zeros((1, N)), np.cumsum(dW, axis=0)), axis=0)
    
    # Evaluate closed form at each timestep for each path
    S = S0 * np.exp((r - 0.5 * sigma**2) * times[:, None] + sigma * W)
    anti_S = S0 * np.exp((r - 0.5 * sigma**2) * times[:, None] + sigma * W * -1)
    
    # Average price for each path
    S_bar = np.mean(S, axis=0)
    anti_S_bar = np.mean(anti_S, axis=0)
    
    # Compute payoffs
    payoffs = np.concatenate([
        np.maximum(S_bar - K, 0),
        np.maximum(anti_S_bar - K, 0)
    ])
    
    # Discount and average
    price = np.exp(-r * T) * np.mean(payoffs)
    
    return price

anti_european = anti_monte_carlo_call(**config)
anti_asian = anti_asian_monte_carlo_call(**config)

print(f" {anti_european}")
print(f" {anti_asian}")

 0.10679516457044363
 0.05743185426893731


If our variance reduction technique works we should see a lower S.D. given the same parameters

Note: We need approx. half as many walks to achieve the same accuracy in our estimate

In [78]:
european_prices = [monte_carlo_call(**config) for _ in range(100)]
anti_european_prices = [anti_monte_carlo_call(**{**config, "N": 5000}) for _ in range(100)]

print(f"European std: {np.std(european_prices)}")
print(f"Antithetic European std: {np.std(anti_european_prices)}")

European std: 0.001450852586696966
Antithetic European std: 0.0014362114984150882
